In [ ]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv()

# Get your API key from .env
api_key1 = os.getenv("GROQ_API_KEY")  
Groq_MODEL = "llama-3.1-8b-instant"

ollama_groq = OpenAI(
    api_key=api_key1,
    base_url="https://api.groq.com/openai/v1"
)

In [ ]:
import requests
requests.get("http://localhost:11434").content
#MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA ="llama3.2:1b"
OLLAMA_BASE_URL="http://localhost:11434/v1"
ollama_local = OpenAI(base_url=OLLAMA_BASE_URL,api_key="ollama")

In [ ]:
links = fetch_website_links("https://www.biman-airlines.com/")

In [ ]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [ ]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a chatbot so that it can ans any question, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
print(get_links_user_prompt("https://www.biman-airlines.com/"))

In [ ]:
def select_relevant_links(url):
    response = ollama_groq.chat.completions.create(
        model=Groq_MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    
    links = json.loads(result)
   
    return links

In [ ]:
print(select_relevant_links("https://www.biman-airlines.com/"))


In [ ]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [ ]:
fetch_page_and_all_relevant_links("https://www.biman-airlines.com/")

In [ ]:
system_prompt = """
You are a specialized Biman Airlines information assistant. 
Your task is to analyze any **public webpage** related to Biman Airlines and extract only **valuable structured information** about:

- Flight schedules and general arrival/departure times  
- Flight delays or late times (without using specific passenger data)  
- Flight status updates (e.g., on-time, delayed)  
- Biman Airlines services, FAQ info, baggage policies, routes, gates, terminals  

Important rules:

1. Never include any **personal passenger information**, flight booking references, or flight numbers tied to specific individuals.  
2. Only use **publicly available data** on the website.  
3. Always output only **JSON format**.  
4. JSON keys should be descriptive, like:
   {
       "route": "Dhaka to Chittagong",
       "departure_time": "10:00",
       "arrival_time": "12:30",
       "delay_minutes": 20,
       "status": "Delayed",
       "service_info": "Baggage allowance 20kg"
   }
5. Do not include any extra text outside JSON.  
6. If some information is missing on the page, use `null` for that field.  
7. Extract **all public flight entries or service info** on the page.  

This prompt ensures you collect **only public flight info, services, and FAQs** from Biman Airlines pages so it can be safely reused for analytics or answering general questions.
"""


In [ ]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a data base.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
get_brochure_user_prompt("Biman Bangladesh","https://www.biman-airlines.com")

In [ ]:
def create_brochure(company_name, url):
    response = ollama_groq.chat.completions.create(
        model=Groq_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [ ]:
create_brochure("Biman Bangladesh","https://www.biman-airlines.com??")

In [ ]:
result="hello"
result+=str(create_brochure("Biman Bangladesh","https://www.biman-airlines.com??"))

In [ ]:
next_prompt = f"""
You have the following Biman Airlines brochure data:



Answer questions or analyze using this data. Be concise and only use the information above.
"""


In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content":str(create_brochure("Biman Bangladesh","https://www.biman-airlines.com")) }] + history + [{"role": "user", "content": Message}]
    stream = ollama_groq.chat.completions.create(model=Groq_MODEL, messages=messages, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()